#### Self-Reflection in RAG using LangGraph, we’ll design a workflow where the agent:

1. Generates an initial answer using retrieved context
2. Reflects on that answer with a dedicated self-critic LLM step
3. If unsatisfied, it can revise the query, retrieve again, or regenerate the answer

In [2]:
import os
from typing import List
from pydantic import BaseModel
from langchain_openai import OpenAIEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langgraph.graph import StateGraph, END
from langfuse.langchain import CallbackHandler
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

langfuse_trace = CallbackHandler()

llm_ollama = init_chat_model(
    model="granite4",
    model_provider="ollama"
)

d:\ai_learning\krisknaik\ultimate-rag-bootcamp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Tho Le\AppData\Local\Temp\ipykernel_21204\2131402494.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
docs = TextLoader("internal_docs.txt", encoding="utf-8").load()
chunks = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
).split_documents(docs)
vectorstore = FAISS.from_documents(chunks, OllamaEmbeddings(
    model="nomic-embed-text"
))
retriever = vectorstore.as_retriever()

In [4]:
# -------------------------
# 2. State Definition
# -------------------------
class RAGReflectionState(BaseModel):
    question: str
    retrieved_docs: List[Document] = []
    answer: str = ""
    reflection: str = ""
    revised: bool = False
    attempts: int = 0

In [5]:
# -------------------------
# 3. Nodes
# -------------------------

# a. Retrieve
def retrieve_docs(state: RAGReflectionState) -> RAGReflectionState:
    docs = retriever.invoke(state.question,  config={"callbacks": [langfuse_trace]})
    return state.model_copy(update={"retrieved_docs": docs})

# b. Generate Answer
def generate_answer(state: RAGReflectionState) -> RAGReflectionState:
    
    context = "\n\n".join([doc.page_content for doc in state.retrieved_docs])
    prompt = f"""
Use the following context to answer the question:

Context:
{context}

Question:
{state.question}
"""
    answer = llm_ollama.invoke(prompt, config={"callbacks": [langfuse_trace]}).content.strip()
    return state.model_copy(update={"answer": answer, "attempts": state.attempts + 1})

In [6]:
# c. Self-Reflect
def reflect_on_answer(state: RAGReflectionState) -> RAGReflectionState:
    
    prompt = f"""
Reflect on the following answer to see if it fully addresses the question. 
State YES if it is complete and correct, or NO with an explanation.

Question: {state.question}

Answer: {state.answer}

Respond like:
Reflection: YES or NO
Explanation: ...
"""
    result = llm_ollama.invoke(prompt, config={"callbacks": [langfuse_trace]}).content
    is_ok = "reflection: yes" in result.lower()
    return state.model_copy(update={"reflection": result, "revised": not is_ok})

In [7]:
# d. Finalizer
def finalize(state: RAGReflectionState) -> RAGReflectionState:
    return state

In [8]:
# -------------------------
# 4. LangGraph DAG
# -------------------------
builder = StateGraph(RAGReflectionState)

builder.add_node("retriever", retrieve_docs)
builder.add_node("responder", generate_answer)
builder.add_node("reflector", reflect_on_answer)
builder.add_node("done", finalize)

builder.set_entry_point("retriever")

builder.add_edge("retriever", "responder")
builder.add_edge("responder", "reflector")
builder.add_conditional_edges(
    "reflector",
    lambda s: "done" if not s.revised or s.attempts >= 2 else "retriever"
)

builder.add_edge("done", END)
graph = builder.compile()

In [9]:
# -------------------------
# 5. Run the Agent
# -------------------------
if __name__ == "__main__":
    user_query = "What are the benefits of agent loop in AI?"
    init_state = RAGReflectionState(question=user_query)
    result = graph.invoke(init_state, config={"callbacks": [langfuse_trace]})

    print("\n🧠 Final Answer:\n", result["answer"])
    print("\n🔁 Reflection Log:\n", result["reflection"])
    print("🔄 Total Attempts:", result["attempts"])


🧠 Final Answer:
 Based on the provided context, there isn’t an explicit discussion about an “agent loop” in AI. The text focuses instead on several related topics such as:

- **Safety:** using Detoxify for toxicity detection, a zero‑shot classifier to filter out‑of‑scope prompts, and red‑team adversarial testing.
- **LLaMA2:** instruction‑tuned with internal support tickets, combined with Retrieval‑Augmented Generation (RAG) for chat Q&A, latency metrics, and evaluation via human judges.
- **Tool‑augmented prompting** that integrates LangGraph, Wikipedia, and SQL search to enable dynamic retrieval‑agent reasoning (e.g., “Give me customer insights from SQL, verify with wiki”).
- **Reformer:** addressing training challenges like bucket collisions, inconsistent loss spikes after 5 k steps, and sparse gradient updates during Locality‑Sensitive Hashing attention.

**Benefits of an Agent Loop in AI (when present):**

1. **Dynamic Decision Making**  
   An agent loop enables the model to ite

In [10]:
if __name__ == "__main__":
    user_query = "What are the transformer variants in production deployments?"
    init_state = RAGReflectionState(question=user_query)
    result = graph.invoke(init_state, config={"callbacks": [langfuse_trace]})

    print("\n🧠 Final Answer:\n", result["answer"])
    print("\n🔁 Reflection Log:\n", result["reflection"])
    print("🔄 Total Attempts:", result["attempts"])


🧠 Final Answer:
 Based on the experiment log, the following transformer models have been deployed in production:

- **EfficientFormer** – Deployed to a Raspberry Pi 4 with quantized int8 mode (top‑1 accuracy of 92.4% on TinyImageNet) and acceptable peak memory usage (290 MB for batch size = 16).

- **TinyBERT** – Used for classifying support‑ticket priorities, achieving an 87 % F1 score with a two‑layer FFN adapter.

No other transformer variants are listed as being in production deployments.

🔁 Reflection Log:
 Reflection: YES  

Explanation: The answer lists the only transformer models that have been deployed in production according to the experiment log—EfficientFormer and TinyBERT. It provides details about their deployment settings (e.g., quantized int8 mode on Raspberry Pi 4, batch size 16) and performance metrics (top‑1 accuracy of 92.4% on TinyImageNet for EfficientFormer; an 87 % F1 score with a two‑layer FFN adapter for TinyBERT). No other transformer variants are mentioned,